# 03 · Model Registry en Unity Catalog — Adult Income

**Objetivo:** elegir el mejor run (de los 5 modelos) por AUC, crear una versión gobernada del modelo y asignarle el alias `Champion`.

> Ejecuta primero los notebooks 01 y 02. Todos los nombres editables están en la primera celda de configuración.


## 1. Configuración

In [0]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

# Detectar el catálogo actual y definir el esquema
current_catalog = spark.catalog.currentCatalog()
mlflow.set_registry_uri("databricks-uc")

CATALOG = current_catalog
SCHEMA = "mlops_income_course"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.income_classifier"

# Apuntar al experimento
mlflow.set_experiment("/Shared/mlops_income_course")

# Inicializar el cliente
client = MlflowClient()

print(f"Catálogo detectado: {CATALOG}")
print(f"Registro de modelos (Unity Catalog): databricks-uc")
print(f"Modelo UC: {MODEL_NAME}")
print(f"Experimento: /Shared/mlops_income_course")


## 2. Seleccionar el mejor run

La regla de promoción es explícita y repetible: mayor AUC entre los 5 modelos entrenados en el notebook 02.


In [0]:
# Validar que el experimento existe
experiment_name = "/Shared/mlops_income_course"
experiment = client.get_experiment_by_name(experiment_name)

if experiment is None:
    raise ValueError(f"El experimento '{experiment_name}' no existe. Ejecuta primero los notebooks 01 y 02.")

experiment_id = experiment.experiment_id
print(f"Experimento encontrado: {experiment_name} (id={experiment_id})")

# Buscar únicamente runs terminados, ordenados por AUC descendente
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.auc DESC"],
    max_results=1,
)

# Validar que hay resultados
if not runs:
    raise ValueError(
        f"No se encontraron runs terminados con métrica AUC en el experimento '{experiment_name}'. "
        "Ejecuta primero los notebooks 01 y 02."
    )

best_run = runs[0]
best_run_id = best_run.info.run_id

best_auc = best_run.data.metrics.get("auc")
print(f"Mejor run: {best_run_id}")
print(f"Nombre: {best_run.data.tags.get('mlflow.runName', '')}")
print(f"AUC: {best_auc:.4f}")


In [0]:
# Mostrar el run ganador con cada dato en una columna separada
# Se construye un DataFrame con tipos homogéneos por columna para evitar errores de Arrow

winner = pd.DataFrame({
    "run_id": [best_run_id],
    "run_name": [best_run.data.tags.get("mlflow.runName", "")],
    "accuracy": [best_run.data.metrics.get("accuracy")],
    "f1": [best_run.data.metrics.get("f1")],
    "auc": [best_run.data.metrics.get("auc")],
})

display(winner)


## 3. Registrar una nueva versión

`catalog.schema.model` es el nombre de tres niveles requerido por Unity Catalog. La firma guardada en el notebook 02 permite validar el esquema de entrada durante inferencia.


In [0]:
model_uri = f"runs:/{best_run_id}/model"

registered_version = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME,
    await_registration_for=300,
)

model_version = registered_version.version

print(f"Modelo registrado: {MODEL_NAME}")
print(f"Versión creada: {model_version}")


## 4. Alias Champion

El alias desacopla a los consumidores de un número de versión. Mover `Champion` a otra versión es una promoción; no exige cambiar el nombre usado por batch inference o por el endpoint de Serving.


In [0]:
# Asignar el alias Champion a la versión recién registrada
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=model_version,
)
print(f"Alias 'Champion' asignado a la versión {model_version} de {MODEL_NAME}")

# Resolver el alias con MlflowClient para confirmar
champion_version = client.get_model_version_by_alias(
    name=MODEL_NAME,
    alias="Champion",
)

print(f"Champion apunta a la versión: {champion_version.version}")


## 5. Smoke test del modelo registrado

Usamos el caso de prueba "candidato califica" (debería predecir 1).


In [0]:
# DataFrame pandas de una fila con el caso de prueba "candidato califica"
sample = pd.DataFrame({
    "age": [45],
    "education_num": [13],
    "hours_per_week": [55],
    "capital_gain": [15000],
    "capital_loss": [0],
    "fnlwgt": [190000],
    "capital_net_ratio": [(15000 - 0) / (55 + 1)],
})

# Cargar el modelo por el alias Champion
model_uri = f"models:/{MODEL_NAME}@Champion"
champion_model = mlflow.pyfunc.load_model(model_uri)

# Ejecutar predict y mostrar la fila junto con la predicción
prediction = champion_model.predict(sample)

sample_with_pred = sample.copy()
sample_with_pred["prediction"] = prediction

display(sample_with_pred)


## Cierre

```text
mejor MLflow run (de 5 modelos comparados) → modelo UC versionado → alias Champion
```

**Comprueba:** abre Catalog Explorer, busca `income_classifier` y revisa versión, alias, firma y lineage. Continúa con el notebook 04.
